## Humann3 analysis

https://huttenhower.sph.harvard.edu/humann/

In [ ]:
salloc -p cpu -t 5:00:00 -c 4
#INSTALLATION
conda create --prefix /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/biobakery3 python=3.7
conda activate /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/biobakery3
#set conda channel priority
conda config --add channels defaults
conda config --add channels bioconda
conda config --add channels conda-forge
conda config --add channels biobakery
conda install humann -c biobakery

#update databases
humann_databases --download utility_mapping full /scratch4/workspace/nikea_ulrich_uml_edu-DR_data/humann_dbs --update-config yes
humann_databases --download chocophlan full /scratch4/workspace/nikea_ulrich_uml_edu-DR_data/humann_dbs --update-config yes
humann_databases --download uniref uniref90_diamond /scratch4/workspace/nikea_ulrich_uml_edu-DR_data/humann_dbs --update-config yes
humann_databases --download uniref uniref50_diamond /scratch4/workspace/nikea_ulrich_uml_edu-DR_data/humann_dbs --update-config yes


In [ ]:
*this script needs to be edited for these samples*
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /scratch3/workspace/nikea_ulrich_uml_edu-gtdb/humann_prof/ofav/slurm-humann-%j.out  # %j = job ID

module load conda/latest
conda activate /scratch3/workspace/nikea_ulrich_uml_edu-gtdb/envs/biobakery3

# Set parameters
SAMPLENAME="ofav"
SAMPLELIST="032024_${SAMPLENAME}_sampleids.txt" 
LISTPATH="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/COL/"
#DB="/scratch3/workspace/nikea_ulrich_uml_edu-gtdb/humann_dbs"
READS="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/COL/assembly/${SAMPLENAME}/repaired"

#work from scratch3 directory (submit script from there)

#concatenate F and R
while IFS= read -r SAMPLEID; do
cat $READS/"${SAMPLEID}"_host_removed_R1.tagged_filter_ready.fastq.gz $READS/"${SAMPLEID}"_host_removed_R2.tagged_filter_ready.fastq.gz > humann_prof/ofav/"${SAMPLEID}"_filter_ready_all.fastq.gz
 if [ $? -eq 0 ]; then
        echo "concatenation successful for sample: $SAMPLEID"
    else
        echo "encountered an error for sample: $SAMPLEID"
        exit 1
    fi
done < "$LISTPATH/${SAMPLELIST}"

#run humann on quality filtered and repaired reads that are concatenated
#run in scratch because the temp files are very big
cd humann_prof/"${SAMPLENAME}"

while IFS= read -r SAMPLEID; do
humann --input "${SAMPLEID}"_filter_ready_all.fastq.gz --output "${SAMPLEID}"_humann --threads 11
 if [ $? -eq 0 ]; then
        echo "humann profile completed for sample: $SAMPLEID"
    else
        echo "humann encountered an error for sample: $SAMPLEID"
        exit 1
    fi 
done < "$LISTPATH/${SAMPLELIST}"

conda deactivate

# JOB-ID: 29436670 (forgot to add more threads so first sample ran before I cancelled and re-ran)
# bash script file name: nikea/COL/bash_scripts/Col_humann_indiv.sh